In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_04_events_mapping
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_04_events_mapping
# Spec Ref  : §1.4 Events Mapping (events_raw → events)
# Purpose   : Map all events from all sources to resolved contact IDs.
#             Unions BQ events (hgi.silver.events) + SF crm_events
#             into one mapped_events table.
#
# Stage-Swap pattern (spec §1.4):
#   1. Write to mapped_events_stage first
#   2. CREATE OR REPLACE mapped_events from stage
#   3. Drop stage
#   Ensures zero downtime — mapped_events is always queryable.
#
# Contact resolution (spec §1.4 — LEFT JOIN):
#   events_raw.source_key_value contains composite ID (bigquery_Event_event_id_...)
#   We resolve to contacts_all.id via two attempts:
#     1. Composite ID match: c.id = concat(source_system, '_Contact_Id_', source_key_value)
#     2. Email match: lower(c.email) = lower(source_key_value)
#   NULL contactid = anonymous event — allowed and expected per spec
#
# Inputs  : hgi.silver.events (BQ), hgi.silver.crm_events (SF Tasks), hgi.silver.contacts_all
# Output  : hgi.silver.mapped_events
#
# Run after: map_01 (contacts_all), map_02 (accounts_all), map_03 (crm_events)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

print(f"=== Map 04: events mapping (events_raw → mapped_events) ===  customer={customer_id}")
print(f"  BQ source  : {sv}.events")
print(f"  SF source  : {sv}.crm_events")
print(f"  Output     : {sv}.mapped_events")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "02b_map_04_events_mapping",
    task_key       = "run_job_D_silver_map",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()

In [0]:
# CELL 3 ── Stage-Swap: write to stage table first
# This guarantees mapped_events is always available for queries
# even while this notebook is running (zero downtime replacement)
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.mapped_events_stage")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {sv}.mapped_events_stage
        USING DELTA
        CLUSTER BY (tenant, meta_event)
        {DELTA_TBLPROPS_MAP}
        AS

        -- ── Part A: BQ native events (from hgi.silver.events = events_raw) ─────
        SELECT
            e.id,
            e.tenant,
            CAST(e.event_timestamp AS TIMESTAMP)    AS event_timestamp,
            -- Normalise BQ event name → meta_event vocabulary
            -- Same rule as crm_events: lowercase + underscore, never null
            COALESCE(
                LOWER(REGEXP_REPLACE(e.event, '[^a-zA-Z0-9]+', '_')),
                'unknown_event'
            )                                        AS meta_event,
            -- Contact resolution attempt 1: composite ID match
            -- attempt 2: email address match (LEFT JOINs so NULLs preserved)
            COALESCE(c1.id, c2.id)                  AS contactid,
            e.domain,
            e.event_text

        FROM {sv}.events e

        -- Attempt 1: source_key_value is a Salesforce contact ID suffix
        LEFT JOIN {sv}.contacts_all c1
            ON c1.id = CONCAT(
                e.source_system, '_Contact_Id_',
                e.source_key_value
            )

        -- Attempt 2: source_key_value is an email address
        LEFT JOIN {sv}.contacts_all c2
            ON LOWER(c2.email) = LOWER(e.source_key_value)
            AND c1.id IS NULL     -- only use c2 when c1 didn't match

        UNION ALL

        -- ── Part B: SF CRM events (from hgi.silver.crm_events) ────────────────
        SELECT
            ce.id,
            ce.tenant,
            ce.event_timestamp,
            ce.meta_event,
            -- contact_id in crm_events is the composite SF ID string
            -- resolve it to the contacts_all row
            COALESCE(c.id, ce.contact_id)           AS contactid,
            c.domain,
            ce.event_text

        FROM {sv}.crm_events ce
        LEFT JOIN {sv}.contacts_all c
            ON c.id = ce.contact_id
    """)
except Exception as e:
    print(f"❌ mapped_events stage build failed: {e}")
    log_audit(
        job_name       = "02b_map_04_events_mapping",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 4 ── Atomic replace: mapped_events always available
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.mapped_events")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {sv}.mapped_events
        USING DELTA
        CLUSTER BY (tenant, meta_event)
        {DELTA_TBLPROPS_MAP}
        AS SELECT * FROM {sv}.mapped_events_stage
    """)

    spark.sql(f"DROP TABLE IF EXISTS {sv}.mapped_events_stage")
except Exception as e:
    print(f"❌ mapped_events atomic replace failed: {e}")
    log_audit(
        job_name       = "02b_map_04_events_mapping",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = f"Atomic replace failed: {str(e)[:450]}",
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 5 ── Spec DQ verification
try:
    total        = cnt(f"{sv}.mapped_events")
    with_contact = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE contactid IS NOT NULL"
    ).collect()[0][0]
    null_meta    = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE meta_event IS NULL"
    ).collect()[0][0]

    contact_pct  = round(100 * with_contact / max(total, 1), 1)
    meta_pct     = round(100 * (total - null_meta) / max(total, 1), 1)

    print(f"  mapped_events: {total:,} rows")
    print(f"  Contact resolution: {with_contact:,}/{total:,} = {contact_pct}%")
    print(f"    Spec gate: ≥{DQ_THRESHOLDS['event_contact_resolution_pct']}%  {'✅ PASS' if contact_pct >= DQ_THRESHOLDS['event_contact_resolution_pct'] else '⚠ BELOW THRESHOLD'}")
    print(f"  meta_event coverage: {meta_pct}%")
    print(f"    Spec gate: 100%  {'✅ PASS' if null_meta == 0 else '⚠ BELOW THRESHOLD'}")
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_04_events_mapping",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise

In [0]:
try:
    total_rows = spark.table(f"{sv}.mapped_events").count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
except Exception as e:
    print(f"❌ Metrics save failed: {e}")
    log_audit(
        job_name       = "02b_map_04_events_mapping",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise